# Using NCI Imaging Data Commons with MONAI

Copyright 2026 Imaging Data Commons

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at http://www.apache.org/licenses/LICENSE-2.0

---

This tutorial demonstrates how to use imaging data from the [NCI Imaging Data Commons (IDC)](https://portal.imaging.datacommons.cancer.gov/) with MONAI for medical imaging AI tasks.

**IDC** is a cloud-based repository of publicly available cancer imaging data with:
- 50+ TB of radiology and pathology images
- No authentication required for access
- AI-generated and expert annotations
- All data in DICOM format

## Setup environment

In [ ]:
!pip install -q monai idc-index itk nibabel itkwasm-dicom

# Restart runtime after installing ITK (required for ITK to load properly)
import sys
if "google.colab" in sys.modules:
    try:
        import itk
    except ImportError:
        print("Restarting runtime to load ITK...")
        import os
        os.kill(os.getpid(), 9)

## Setup imports

In [ ]:
import os
import tempfile

import torch
import numpy as np
import matplotlib.pyplot as plt
from idc_index import IDCClient

from monai.transforms import (
    Compose,
    LoadImaged,
    EnsureChannelFirstd,
    Orientationd,
    Spacingd,
    ScaleIntensityRanged,
)
from monai.data import Dataset, DataLoader
from monai.data.image_reader import ITKReader
import monai

monai.config.print_config()

## 1. Query IDC Data

Use `idc-index` to query IDC's metadata index using SQL.

In [ ]:
# Initialize IDC client
client = IDCClient()

# Check IDC version and data scale
print(f"IDC version: {client.get_idc_version()}")

stats = client.sql_query("""
    SELECT COUNT(DISTINCT collection_id) as collections,
           COUNT(DISTINCT PatientID) as patients
    FROM index
""")
print(f"Collections: {stats.iloc[0]['collections']}, Patients: {stats.iloc[0]['patients']}")

In [ ]:
# Find lung CT collections by joining collections_index with index
client.fetch_index("collections_index")

# Join to get modality information (not available in collections_index directly)
lung_collections = client.sql_query("""
    SELECT c.collection_id, c.Subjects, c.CancerTypes,
           COUNT(DISTINCT CASE WHEN i.Modality = 'CT' THEN i.SeriesInstanceUID END) as ct_series
    FROM collections_index c
    JOIN index i ON c.collection_id = i.collection_id
    WHERE c.CancerTypes LIKE '%Lung%'
    GROUP BY c.collection_id, c.Subjects, c.CancerTypes
    HAVING ct_series > 0
    ORDER BY c.Subjects DESC
    LIMIT 5
""")
print("Lung CT collections:")
print(lung_collections.to_string(index=False))

In [ ]:
# Query specific CT series (using small rider_pilot collection for demo)
series_df = client.sql_query("""
    SELECT SeriesInstanceUID, PatientID, Modality,
           ROUND(series_size_MB, 2) as size_mb
    FROM index
    WHERE collection_id = 'rider_pilot' AND Modality = 'CT'
    LIMIT 3
""")
print(f"Found {len(series_df)} CT series")

## 2. Download DICOM Data

In [ ]:
# Download to temporary directory
data_dir = tempfile.mkdtemp(prefix="idc_monai_")
series_uids = list(series_df['SeriesInstanceUID'])

print(f"Downloading {len(series_uids)} series...")
client.download_from_selection(
    seriesInstanceUID=series_uids,
    downloadDir=data_dir,
    dirTemplate="%SeriesInstanceUID"
)
print("Done!")

## 3. Load with MONAI Transforms

MONAI's `LoadImaged` with `ITKReader` directly loads DICOM series from directories.

In [ ]:
# Define transforms for CT preprocessing
# Use ITKReader explicitly to load DICOM series from directories
transforms = Compose([
    LoadImaged(keys=["image"], reader=ITKReader()),
    EnsureChannelFirstd(keys=["image"]),
    Orientationd(keys=["image"], axcodes="RAS"),
    Spacingd(keys=["image"], pixdim=(1.5, 1.5, 2.0)),
    ScaleIntensityRanged(keys=["image"], a_min=-175, a_max=250,
                         b_min=0.0, b_max=1.0, clip=True),
])

# Create dataset
data_dicts = [{"image": os.path.join(data_dir, uid)} for uid in series_uids]
dataset = Dataset(data=data_dicts, transform=transforms)

# Load sample
sample = dataset[0]
print(f"Image shape: {sample['image'].shape}")
print(f"Value range: [{sample['image'].min():.2f}, {sample['image'].max():.2f}]")

## 4. Visualize

In [ ]:
image = sample['image'][0]
z = image.shape[2] // 2

plt.figure(figsize=(6, 6))
plt.imshow(image[:, :, z].T, cmap='gray', origin='lower')
plt.title(f'CT from IDC (slice {z})')
plt.axis('off')
plt.show()

## 5. Loading DICOM Segmentations

IDC contains DICOM Segmentation (DICOM-SEG) objects. These are different from regular DICOM series:
- **Enhanced multiframe**: All slices in a single file
- **Cannot use ITKReader**: Standard readers don't support this format

We use `itkwasm-dicom` which wraps dcmqi for robust DICOM-SEG reading.

In [ ]:
# Fetch seg_index to find paired image-segmentation data
client.fetch_index("seg_index")

# Find CT with TotalSegmentator segmentations (104 anatomical structures)
paired = client.sql_query("""
    SELECT src.SeriesInstanceUID as image_uid,
           seg.SeriesInstanceUID as seg_uid,
           src.collection_id, seg.total_segments,
           ROUND(src.series_size_MB, 2) as image_mb
    FROM seg_index seg
    JOIN index src ON seg.segmented_SeriesInstanceUID = src.SeriesInstanceUID
    WHERE src.Modality = 'CT'
      AND seg.AlgorithmName LIKE '%TotalSegmentator%'
    ORDER BY src.series_size_MB ASC
    LIMIT 3
""")
print("CT with TotalSegmentator segmentations:")
print(paired.to_string(index=False))

In [ ]:
# Download one image-segmentation pair
demo_pair = paired.iloc[0]
seg_dir = tempfile.mkdtemp(prefix="idc_seg_")

print(f"Downloading image and segmentation pair...")
client.download_from_selection(
    seriesInstanceUID=[demo_pair['image_uid'], demo_pair['seg_uid']],
    downloadDir=seg_dir,
    dirTemplate="%SeriesInstanceUID"
)
print("Done!")

In [ ]:
from pathlib import Path
from itkwasm_dicom import read_segmentation
from monai.data import MetaTensor

def load_dicom_seg(seg_dir_path):
    """Load DICOM-SEG file using ITKWasm.
    
    Returns MetaTensor with array in (X, Y, Z) order to match ITKReader.
    """
    seg_path = Path(seg_dir_path)
    # Find .dcm file in directory
    dcm_files = list(seg_path.glob("*.dcm"))
    if not dcm_files:
        raise FileNotFoundError(f"No .dcm files in {seg_path}")
    
    # Read with ITKWasm
    seg_image, overlay_info = read_segmentation(dcm_files[0])
    
    # ITKWasm returns array in (Z, Y, X) order but metadata in (X, Y, Z) order
    # Transpose array to (X, Y, Z) to match metadata and ITKReader convention
    seg_array = np.asarray(seg_image.data)
    seg_array = np.transpose(seg_array, (2, 1, 0))  # (Z, Y, X) -> (X, Y, Z)
    
    # Build affine matrix from spatial metadata (already in X, Y, Z order)
    spacing = np.array(seg_image.spacing)
    origin = np.array(seg_image.origin)
    direction = np.array(seg_image.direction).reshape(3, 3)
    
    affine = np.eye(4)
    affine[:3, :3] = direction @ np.diag(spacing)
    affine[:3, 3] = origin
    
    # Create MONAI MetaTensor with affine
    meta_tensor = MetaTensor(seg_array)
    meta_tensor.affine = affine
    
    return meta_tensor, overlay_info

# Load segmentation
seg_path = os.path.join(seg_dir, demo_pair['seg_uid'])
seg_tensor, overlay_info = load_dicom_seg(seg_path)

print(f"Segmentation shape: {seg_tensor.shape}")
print(f"Unique labels: {np.unique(seg_tensor)[:10]}...")  # First 10 labels
print(f"Affine matrix:\n{seg_tensor.affine}")

In [ ]:
# Load CT and verify shapes match
image_path = os.path.join(seg_dir, demo_pair['image_uid'])
ct_transforms = Compose([
    LoadImaged(keys=["image"], reader=ITKReader()),
    EnsureChannelFirstd(keys=["image"]),
])
ct_data = ct_transforms({"image": image_path})
ct_image = ct_data["image"][0].numpy()  # Remove channel dim

print(f"CT image shape: {ct_image.shape}")
print(f"Segmentation shape: {seg_tensor.shape}")

# Check orientation codes match
import nibabel as nib
ct_axcodes = nib.aff2axcodes(ct_data['image'].affine)
seg_axcodes = nib.aff2axcodes(seg_tensor.affine)
print(f"\nCT orientation: {ct_axcodes}")
print(f"SEG orientation: {seg_axcodes}")

In [ ]:
# Reorient both to same orientation (RAS) for visualization
from monai.transforms import Orientation

# Reorient CT to RAS
ct_reorient = Orientation(axcodes="RAS")
ct_ras = ct_reorient(ct_data["image"])[0].numpy()

# Reorient SEG to RAS (need to add channel dim first)
seg_with_channel = seg_tensor.unsqueeze(0)
seg_ras = ct_reorient(seg_with_channel)[0].numpy()

print(f"CT (RAS) shape: {ct_ras.shape}")
print(f"SEG (RAS) shape: {seg_ras.shape}")

# Visualize
z_slices_with_labels = np.where(seg_ras.sum(axis=(0, 1)) > 0)[0]
if len(z_slices_with_labels) > 0:
    z_mid = z_slices_with_labels[len(z_slices_with_labels) // 2]
else:
    z_mid = ct_ras.shape[2] // 2

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# CT image only
axes[0].imshow(ct_ras[:, :, z_mid].T, cmap='gray', origin='lower')
axes[0].set_title('CT Image')
axes[0].axis('off')

# Segmentation only
axes[1].imshow(seg_ras[:, :, z_mid].T, cmap='nipy_spectral', origin='lower')
axes[1].set_title(f'Segmentation ({int(seg_ras.max())} labels)')
axes[1].axis('off')

# Overlay
axes[2].imshow(ct_ras[:, :, z_mid].T, cmap='gray', origin='lower')
seg_slice = seg_ras[:, :, z_mid]
mask = np.ma.masked_where(seg_slice == 0, seg_slice)
axes[2].imshow(mask.T, cmap='nipy_spectral', alpha=0.5, origin='lower')
axes[2].set_title('Overlay')
axes[2].axis('off')

plt.suptitle(f'CT + TotalSegmentator Segmentation (slice {z_mid}, RAS orientation)', fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nSegmentation loaded successfully with ITKWasm!")
print(f"Both CT and segmentation reoriented to RAS for visualization.")

## 6. Check Licenses

In [ ]:
# Always check licenses before use
uid_list = ", ".join(f"'{uid}'" for uid in series_uids)
licenses = client.sql_query(f"""
    SELECT license_short_name, COUNT(*) as count
    FROM index WHERE SeriesInstanceUID IN ({uid_list})
    GROUP BY license_short_name
""")
print("Licenses:")
print(licenses.to_string(index=False))

## 7. Cleanup

In [ ]:
# import shutil; shutil.rmtree(data_dir)  # Uncomment to delete
print(f"Data at: {data_dir}")

## Summary

This tutorial demonstrated:
1. Querying IDC with `idc-index` SQL interface
2. Downloading DICOM data (no authentication needed)
3. Loading CT images into MONAI with `LoadImaged` and `ITKReader`
4. Finding paired segmentations via `seg_index`
5. Loading DICOM-SEG files with `itkwasm-dicom` (standard readers don't support this format)

**Key Point**: DICOM-SEG requires special handling - use `itkwasm-dicom.read_segmentation()` instead of ITKReader.

**Resources:**
- [IDC Portal](https://portal.imaging.datacommons.cancer.gov/)
- [IDC Documentation](https://learn.canceridc.dev/)
- [idc-index](https://github.com/ImagingDataCommons/idc-index)
- [IDC-MONAI toolkit](https://github.com/ImagingDataCommons/idc-monai)
- [ITKWasm DICOM](https://wasm.itk.org/en/latest/introduction/file_formats/dicom.html)